A1 — clone + patch



In [1]:
%%bash
set -e
mkdir -p /kaggle/temp
cd /kaggle/temp && rm -rf FreeFine && git clone -q https://github.com/CIawevy/FreeFine.git
python3 - <<'PY'
import pathlib
root = pathlib.Path("/kaggle/temp/FreeFine/evaluation/metrics")
p = root/"main.py"; p.write_text(p.read_text().replace("args.3d","getattr(args, '3d')"))
for f in [root/"MD"/"mean_distance.py", root/"MD"/"dift_sd.py"]:
    f.write_text(f.read_text().replace("stabilityai/stable-diffusion-2-1","sd2-community/stable-diffusion-2-1"))
print("patched")
PY

patched


A2 — metric_env



In [2]:
%%bash
set -e
pip install -q --root-user-action=ignore uv
uv python install 3.10.13
VENV=/kaggle/temp/metric_env; PY=$VENV/bin/python; REPO=/kaggle/temp/FreeFine
rm -rf $VENV && uv venv --python 3.10.13 $VENV
uv pip install --python $PY torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124
uv pip install --python $PY "setuptools<70" wheel pip
grep -vi '^clip' $REPO/evaluation/metrics/requirements.txt > /tmp/metric_req.txt
uv pip install --python $PY -r /tmp/metric_req.txt
uv pip install --python $PY "setuptools<70"
uv pip install --python $PY --no-build-isolation "clip @ git+https://github.com/openai/CLIP.git@dcba3cb2e2827b402d2701e7e1c7d9fed8a20ef1"
uv pip install --python $PY "pyarrow<16" "datasets<3"
wget -q https://dl.fbaipublicfiles.com/mmf/clip/bpe_simple_vocab_16e6.txt.gz -P /tmp
for d in $(find $VENV -path '*/site-packages/clip' -o -path '*open_clip' -type d); do cp /tmp/bpe_simple_vocab_16e6.txt.gz "$d/" 2>/dev/null || true; done
echo "metric_env ready -> datasets $($PY -c 'import datasets; print(datasets.__version__)')"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 50.2 MB/s eta 0:00:00
metric_env ready -> datasets 2.21.0


 Downloaded cpython-3.10.13-linux-x86_64-gnu (download)
Installed Python 3.10.13 in 1.31s
 + cpython-3.10.13-linux-x86_64-gnu (python3.10)
Using CPython 3.10.13
Creating virtual environment at: /kaggle/temp/metric_env
Activate with: source /kaggle/temp/metric_env/bin/activate
Using Python 3.10.13 environment at: /kaggle/temp/metric_env
Resolved 27 packages in 315ms
 Downloaded nvidia-cuda-cupti-cu12
 Downloaded torchaudio
 Downloaded nvidia-cuda-nvrtc-cu12
 Downloaded nvidia-nvjitlink-cu12
 Downloaded torchvision
 Downloaded nvidia-curand-cu12
 Downloaded pillow
 Downloaded networkx
 Downloaded triton
 Downloaded nvidia-cusolver-cu12
 Downloaded nvidia-cusparselt-cu12
 Downloaded numpy
 Downloaded nvidia-nccl-cu12
 Downloaded nvidia-cufft-cu12
 Downloaded nvidia-cusparse-cu12
 Downloaded sympy
 Downloaded nvidia-cublas-cu12
 Downloaded nvidia-cudnn-cu12
 Downloaded torch
Prepared 27 packages in 47.13s
Installed 27 packages in 305ms
 + filelock==3.29.0
 + fsspec==2026.4.0
 + jinja2==3.1

A3 — GeoBench-2D 



In [3]:
import os, glob, shutil
GEO="/kaggle/temp/GeoBenchMeta"; os.makedirs(f"{GEO}/Geo-Bench-2D", exist_ok=True)
shutil.copy(glob.glob("/kaggle/input/*/**/annotation_2d.json", recursive=True)[0], f"{GEO}/annotation_2d.json")
linked=set()
for sub in glob.glob("/kaggle/input/*/**/Geo-Bench-2D/*", recursive=True):
    if not os.path.isdir(sub): continue
    name=os.path.basename(sub); dst=f"{GEO}/Geo-Bench-2D/{name}"
    if name in linked: continue
    if os.path.islink(dst): os.remove(dst)
    elif os.path.exists(dst): shutil.rmtree(dst)
    os.symlink(sub, dst); linked.add(name)
print("linked subfolders:", sorted(linked), flush=True)   # should include coarse_img

linked subfolders: ['coarse_img', 'source_img', 'source_img_full_v2', 'source_mask', 'target_mask']


A4 — symlink gen images + build metadata + build 6 manifests (self-contained)



In [4]:
import os, glob, json, math, csv, shutil
import numpy as np
from PIL import Image
GEO="/kaggle/temp/GeoBenchMeta"
DS ="/kaggle/input/datasets/georgiostzamouranis/freefine-geobench2d-bggen"   # confirm
PNG_ROOT=os.path.join(DS,"gen_results_2d_final","gen_results_2d_backup")
assert os.path.isdir(PNG_ROOT), f"fix DS: {PNG_ROOT}"
link=os.path.join(GEO,"Gen_results_FreeFine_2d")
if os.path.islink(link): os.remove(link)
elif os.path.isdir(link): shutil.rmtree(link)
os.symlink(PNG_ROOT, link)
print("gen images:", len(glob.glob(f"{link}/**/*.png", recursive=True)), flush=True)

ann=json.load(open(f"{GEO}/annotation_2d.json"))
def load_mask(rel): return np.array(Image.open(os.path.join(GEO,rel)).convert("L"))>127
rows=[]; sc={}
for d,da in ann.items():
    for i,ins in da["instances"].items():
        sk=(d,i)
        for c,lf in ins.items():
            dx,dy,dz,rx,ry,rz,sx,sy,sz=lf["edit_param"]
            if sk not in sc: sc[sk]=load_mask(lf["ori_mask_path"])
            src=sc[sk]; H,W=src.shape; diag=math.hypot(H,W)
            tgt=load_mask(lf["tgt_mask_path"])
            if tgt.shape!=src.shape:
                tgt=np.array(Image.fromarray(tgt.astype(np.uint8)*255).resize((W,H),Image.NEAREST))>127
            inter=int(np.logical_and(src,tgt).sum()); union=int(np.logical_or(src,tgt).sum())
            is_m=abs(dx)>1e-6 or abs(dy)>1e-6; is_r=abs(rz)>1e-6; is_s=abs(sx-1)>1e-6 or abs(sy-1)>1e-6
            ts=[t for t,f in [("move",is_m),("rotate",is_r),("resize",is_s)] if f]
            et=ts[0] if len(ts)==1 else ("none" if not ts else "mixed")
            rows.append(dict(da_n=d,ins_id=i,case_id=c,edit_type=et,
                norm_translation=math.hypot(dx,dy)/diag, rotation_deg=abs(rz),
                scale_change=(abs(math.log(sx))+abs(math.log(sy))) if sx>0 and sy>0 else 0.0,
                mask_area_ratio=int(src.sum())/(H*W), overlap_iou=inter/union if union else 0.0,
                gen_rel=f"Gen_results_FreeFine_2d/{d}/{i}/{c}.png"))
def mm(k):
    v=np.array([r[k] for r in rows],float); return (v-v.min())/(v.max()-v.min()+1e-9)
D=0.25*(mm("norm_translation")+mm("rotation_deg")+mm("scale_change")+mm("mask_area_ratio"))
q1,q2=np.quantile(D,[1/3,2/3])
for r,dv in zip(rows,D): r["difficulty"]="easy" if dv<q1 else ("medium" if dv<q2 else "hard")
with open("/kaggle/working/sample_metadata.csv","w",newline="") as f:
    w=csv.DictWriter(f,fieldnames=list(rows[0].keys())); w.writeheader(); w.writerows(rows)

os.makedirs("/kaggle/temp/manifests",exist_ok=True)
def build(pred,name):
    keep={(r["da_n"],r["ins_id"],r["case_id"]) for r in rows if pred(r)}
    out={}
    for d,da in ann.items():
        insts={}
        for i,ins in da["instances"].items():
            cs={}
            for c,leaf in ins.items():
                if (d,i,c) in keep:
                    lf=dict(leaf); lf["gen_img_path"]=f"Gen_results_FreeFine_2d/{d}/{i}/{c}.png"; cs[c]=lf
            if cs: insts[i]=cs
        if insts:
            nd={k:v for k,v in da.items() if k!="instances"}; nd["instances"]=insts; out[d]=nd
    json.dump(out,open(f"/kaggle/temp/manifests/manifest_{name}.json","w"))
    n=sum(len(c) for da in out.values() for c in da["instances"].values()); print(f"{name}: {n}",flush=True)
for et in ["move","rotate","resize"]: build(lambda r,e=et:r["edit_type"]==e,f"type_{et}")
for df in ["easy","medium","hard"]:   build(lambda r,f=df:r["difficulty"]==f,f"diff_{df}")
print("manifests ready",flush=True)

gen images: 5677
type_move: 1439
type_rotate: 1603
type_resize: 2635
diff_easy: 1892
diff_medium: 1892
diff_hard: 1893
manifests ready


A5 — run the 6 groups → save results



In [5]:
import os, re, glob, json, time, subprocess
GEO="/kaggle/temp/GeoBenchMeta"; MET="/kaggle/temp/FreeFine/evaluation/metrics"
PY="/kaggle/temp/metric_env/bin/python"
TASK="100111011"   # FID + BGC + SUBC + WRAP_E + FID_DINO + FID_KD
env=os.environ.copy(); env.update({"MPLBACKEND":"Agg","PYTHONUNBUFFERED":"1",
    "HF_HOME":"/kaggle/temp/hf","TORCH_HOME":"/kaggle/temp/torch",
    "PYTORCH_CUDA_ALLOC_CONF":"expandable_segments:True"})
results={}
for man in sorted(glob.glob("/kaggle/temp/manifests/manifest_*.json")):
    name=os.path.basename(man).replace("manifest_","").replace(".json","")
    print(f"\n[{name}] running...", flush=True); t0=time.time()
    cmd=[PY,"main.py","--path",man,"--use_relative_path","--base_dir",GEO,
         "--fid_path",f"{GEO}/Geo-Bench-2D/source_img_full_v2","--task",TASK,"--level","0"]
    out=subprocess.run(cmd,cwd=MET,env=env,capture_output=True,text=True)
    tail=out.stdout+out.stderr; dt=int(time.time()-t0); vals={}
    for key in ["FID_DINO","FID_KD","FID","SUBC","BGC","WRAP_E"]:
        m=re.findall(rf"{key}:\s*([-\d.eE]+)", tail)
        if m: vals[key]=round(float(m[-1]),4)
    results[name]=vals
    print(f"[{name}] {dt//60}m{dt%60}s -> {vals}", flush=True)
    if not vals: print("  !! parse failed tail:\n", tail[-1500:], flush=True)
json.dump(results, open("/kaggle/working/phase1_group_metrics.json","w"), indent=2)
print("\n===== SUMMARY =====\n", json.dumps(results, indent=2))


[diff_easy] running...
[diff_easy] 8m46s -> {'FID_DINO': 772.0728, 'FID_KD': 0.1909, 'FID': 57.585, 'SUBC': 0.9417, 'BGC': 0.9709, 'WRAP_E': 0.0462}

[diff_hard] running...
[diff_hard] 8m27s -> {'FID_DINO': 533.7684, 'FID_KD': 0.167, 'FID': 42.174, 'SUBC': 0.865, 'BGC': 0.9641, 'WRAP_E': 0.0503}

[diff_medium] running...
[diff_medium] 8m17s -> {'FID_DINO': 589.3164, 'FID_KD': 0.1453, 'FID': 44.0786, 'SUBC': 0.9274, 'BGC': 0.9661, 'WRAP_E': 0.0455}

[type_move] running...
[type_move] 6m57s -> {'FID_DINO': 580.8259, 'FID_KD': 0.1297, 'FID': 47.397, 'SUBC': 0.9592, 'BGC': 0.967, 'WRAP_E': 0.0489}

[type_resize] running...
[type_resize] 10m19s -> {'FID_DINO': 531.2929, 'FID_KD': 0.1416, 'FID': 40.8476, 'SUBC': 0.8916, 'BGC': 0.9669, 'WRAP_E': 0.049}

[type_rotate] running...
[type_rotate] 7m26s -> {'FID_DINO': 588.7471, 'FID_KD': 0.142, 'FID': 47.5545, 'SUBC': 0.9008, 'BGC': 0.9673, 'WRAP_E': 0.0433}

===== SUMMARY =====
 {
  "diff_easy": {
    "FID_DINO": 772.0728,
    "FID_KD": 0.1909,
